# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import os
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
logging.info(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
logging.info(f"✅ Torch CUDA available: {cuda_test}")
device_name = torch.cuda.get_device_name(0)
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"🖥️ Device Name: {device_name} | Device reference: {torch_device.type}")

### machine learning (scikit-learn)
import math
import pprint
from typing import cast
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings(
    "ignore",
    message="mtime may not be reliable on this filesystem, falling back to numerical ordering"
)
from transformers import set_seed, Trainer  # type: ignore
from tsfm_public import (
    TimeSeriesForecastingPipeline,
)
from tsfm_public.toolkit.time_series_preprocessor import get_datasets, prepare_data_splits
from tsfm_public.toolkit.visualization import plot_predictions
from tsfm_public.toolkit.service_util import save_deployment_package

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.preprocessing_project_specific as pps
import smartcheck.deep_learning_project_specific as dlps
import smartcheck.modeling_project_specific as mps

# 2. Loading and Preprocessing

## 2.1 Loading data

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Data Refactoring pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    # "date_et_heure_de_comptage",
    "date_et_heure_de_comptage_local",
    # "date_et_heure_de_comptage_utc",
    "orientation_compteur",
    # "latitude",
    # "longitude",
    # "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    # "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("convert_datetime", pps.DatetimePreprocessingTransformer(timestamp_col="date_et_heure_de_comptage",
                                                              for_sarimax=True)),
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

# 3. Modeling Training and Prediction

## 3.1 Contextual variables

#### For all experiments (common)

In [ ]:
OUT_DIR = "ttm_results.model" # model runtime and training data archiving/export.
# Parameter for the model
context_length = 512 # max possible for this model
prediction_length = 96 # max possible for this model
# Parameters for the trainings
fcm_context_length = 256 # 1 lag = 1 hour
learning_rate: float = 0.0002
num_epochs: int = 200
patience: int = 40
batch_size: int = 64
# Keep the same randoom seed for better reproducibility and evaluation
set_seed(42)
# Preprocessor general condition (keep homogeneous conditions)
fewshot_fraction = 1 # fasten training by returning a percent original train dataset
split_config = {"train": 0.6, "test": 0.25}
timestamp_column = "date_et_heure_de_comptage_local"
target_columns = ["comptage_horaire"]
column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": [],
    "target_columns": target_columns,
    "categorical_columns": [
        "vacances_scolaires",
        "weather_code_wmo_code_category",
    ],
    "control_columns": [
    ],
    "observable_columns": [
        "jour_ferie",
        "vacances_scolaires",
        "temperature_2m_c",
        "rain_mm",
        "snowfall_cm",
        "weather_code_wmo_code_category",
    ],
}

#### For each specific experiments

In [ ]:
dict_compteurs = {
    # "experiment_1": {
    #     "key": ('135 avenue Daumesnil','SE-NO'),
    #     "name": "Daumesnil_S-N",
    #     "sub_range": (0,),
    #     "steps_per_epoch": math.ceil(5900 / batch_size),
    #     ### Context backup
    #     "out_dir": OUT_DIR,      
    #     "context_length": context_length,
    #     "fcm_context_length": fcm_context_length,
    #     "prediction_length": prediction_length,
    #     ### Manual override for training is finished
    #     # "best_checkpoint": "checkpoint-34",
    # },
    # "experiment_2": {
    #     "key": ('102 boulevard de Magenta', 'SE-NO'),
    #     "name": "Magenta_SE-NO",
    #     "sub_range": (0,),
    #     "steps_per_epoch": math.ceil(5900 / batch_size),
    #     ### Context backup
    #     "out_dir": OUT_DIR,            
    #     "context_length": context_length,
    #     "fcm_context_length": fcm_context_length,
    #     "prediction_length": prediction_length,
    #     ### Manual override for training is finished
    #     # "best_checkpoint": "checkpoint-7",
    # },
    "experiment_3": {
        "key": ('Totem 73 boulevard de Sébastopol', 'S-N'),
        "name": "Sébastopol_S-N_fcm256_bs64",
        "sub_range": (0,),
        "steps_per_epoch": math.ceil(5724 / batch_size),
        ### Context backup
        "out_dir": OUT_DIR,             
        "context_length": context_length,
        "fcm_context_length": fcm_context_length,
        "prediction_length": prediction_length,
        ### Manual override for training is finished
        "best_checkpoint": "checkpoint-200",
    },
}

## 3.2 Data Viz of Time Series per counter

In [ ]:
# splitted dataframe per counter
grouped_df = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for experiment, exp_params in dict_compteurs.items():
    key = exp_params["key"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {exp_params} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue

    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]

    fig, axs = plt.subplots(len(target_columns), 1, figsize=(10, 2 * len(target_columns)), squeeze=False)
    for ax, target_column in zip(axs, target_columns):
        ax[0].plot(df_compteur_sub[timestamp_column], df_compteur_sub[target_column])
    plt.show()

## 3.3 Transfert learning with preprocessing and training

#### Définition et ajustement du modèle granite pour finetuning avec gel des couches pré-entrainées
> Environ 500k paramètres gelés mais il reste ceux ajoutés via notre contexte de variables exogènes, le nombre de paramètres additionnels dépend des variables exogènes et de la fenetre de contexte (FCM)
> - 24 lags (1 jour) -> environ 900k paramètres
> - 48 lags (2 jours) -> environ 2,7M paramètres
> - 168 lags (7 jours) -> environ 29M de paramètres
> - 256 lags (10,6 jours) -> environ 68M de paramètres
> - 512 lags (21,3 jours) -> environ 271M de paramètres

#### Boucle d'entrainement du modèle

In [ ]:
# splitted dataframe per counter
grouped_df = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

model_results = {}
# Make a forecast on the target column given the input data.
for experiment, exp_params in dict_compteurs.items():
    # Filtering experiment and associated dataset
    key = exp_params["key"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {pprint.pformat(exp_params)} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue
    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]

    # Definition and model fine tuning
    name = exp_params["name"]
    steps_per_epoch = exp_params["steps_per_epoch"]
    deploy_dir = os.path.join(OUT_DIR, f"{name}_deploy")
    preproc_dir = os.path.join(OUT_DIR, f"{name}_preproc")
    output_dir = os.path.join(OUT_DIR, f"{name}_output")
    logging_dir = os.path.join(OUT_DIR, f"{name}_log")
    (
        tsp,
        model,
        args,
        optimizer,
        scheduler,
        early_stop_cb,
        tracking_cb
    ) = dlps.fine_tune_model(
        output_dir,
        logging_dir,
        context_length,
        prediction_length,
        fcm_context_length,
        column_specifiers,
        learning_rate,
        num_epochs,
        batch_size,
        steps_per_epoch,
        patience,
        torch_device.type,        
    )

    # Split train, valid, test for dataframes and datasets for training
    df_train, df_valid, df_test = prepare_data_splits(  # type: ignore
        df_compteur_sub,
        context_length=context_length,
        split_config=split_config  # type: ignore
    )
    logging.info(f"Dataframe lengths: train = {len(df_train)}, val = {len(df_valid)}, test = {len(df_test)}")
    dataset_train, dataset_valid, dataset_test = get_datasets(  # type: ignore
        tsp,
        df_compteur_sub,
        split_config,  # type: ignore
        stride=prediction_length,
        fewshot_fraction=fewshot_fraction,
        fewshot_location="first",
        use_frequency_token=model.config.resolution_prefix_tuning,
    )
    logging.info(f"Dataset batch lengths: train = {len(dataset_train)}, val = {len(dataset_valid)}, test = {len(dataset_test)}")

    # Définition du modèle et de son trainer
    finetune_forecast_trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset_train,
        eval_dataset=dataset_valid,
        callbacks=[early_stop_cb, tracking_cb],
        optimizers=(optimizer, scheduler)  # type: ignore
    )

    # Priorité au reentrainement à partir du best checkpoint
    best_checkpoint = dlps.train_or_resume(finetune_forecast_trainer, exp_params)

    # sauvegarde des résultats en mémoire
    model_results[experiment] = {
        "exp_params": exp_params,
        "df_train": df_train,
        "df_valid": df_valid,
        "df_test": df_test,
        "best_checkpoint": best_checkpoint,
        "model": model,
        "tsp": tsp,
    }

    # sauvegarde sur disque de l'experience via joblib / yaml (preprocessor inclus)
    dlps.save_granite_model(experiment, model_results[experiment])
    dlps.save_preprocessor_state(tsp, preproc_dir)
    save_deployment_package(deploy_dir, model, ts_processor=tsp)

## 3.3 Predictions

#### From memory context

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    exp_params = model_result["exp_params"]
    df_train = model_result["df_train"]
    df_valid = model_result["df_valid"]
    df_test = model_result["df_test"]
    model = model_result["model"]
    tsp = model_result["tsp"]
    logging.info(f"\n\nParamètres {exp_params} :\n")
    name = exp_params["name"]
    out_dir = exp_params["out_dir"]
    model.to(torch_device)  # type: ignore
    
    # Create the evaluation pipeline
    pipeline = TimeSeriesForecastingPipeline(
        model=model,
        device=torch_device,
        feature_extractor=tsp,
        batch_size=batch_size,
    )

    # Collect the test df and run the predictions
    predictions_df_train = cast(pd.DataFrame, pipeline(df_train))
    predictions_df_test = cast(pd.DataFrame, pipeline(df_test))

    # Print/Plot the predictions
    plot_predictions(
        input_df=df_test,
        predictions_df=predictions_df_test,
        freq="h",
        timestamp_column=timestamp_column,
        channel=target_columns[0],
        # we check the prediction on each of the 3 previous week and in the future
        indices=[ -24*7*3, -24*7*2, -24*7*1, -1],
        num_plots=4,
    )
    plt.show()

    ###############################################
    ### Stockage des données pour les métriques ###
    ###############################################
    model_results[experiment]["y_train"] = predictions_df_train.comptage_horaire.apply(
        lambda x: x[0]
    )[:-1]
    model_results[experiment]["y_train_pred"] = predictions_df_train.comptage_horaire_prediction.apply(
        lambda x: x[0]
    )[:-1]
    model_results[experiment]["y_test"] = predictions_df_test.comptage_horaire.apply(
        lambda x: x[0]
    )[:-1]
    model_results[experiment]["y_test_pred"] = predictions_df_test.comptage_horaire_prediction.apply(
        lambda x: x[0]
    )[:-1]
    model_results[experiment]["dates_test"] = predictions_df_test[["date_et_heure_de_comptage_local"]][:-1]

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    key = model_result["exp_params"]["key"]
    name = model_result["exp_params"]["name"]
    best_checkpoint = model_result["best_checkpoint"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {key} {name} {best_checkpoint} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue
    checkpoint_dir = os.path.join(out_dir, f"{name}_output", best_checkpoint)
    preproc_dir = os.path.join(out_dir, f"{name}_preproc")

    y_train = model_result["y_train"]
    y_train_pred = model_result["y_train_pred"]
    y_test = model_result["y_test"]
    y_test_pred = model_result["y_test_pred"]
    dates_test = model_result["dates_test"]
    periode_limite = (
        dates_test.date_et_heure_de_comptage_local[max(dates_test.index.max()-24*7*4,
                                                        dates_test.index.min())],
        dates_test.date_et_heure_de_comptage_local[dates_test.index.max()-1],
    )

    # Affichage des metrique train et test
    model_train_metrics = mps.compute_metrics(
        y_train,
        y_train_pred,
    )
    model_test_metrics = mps.compute_metrics(
        y_test,
        y_test_pred
    )
    logging.info(f"Metriques du modèle (Train): {model_train_metrics}")
    logging.info(f"Metriques du modèle (Test): {model_test_metrics}")

    # projection des predictions de test dans le temps
    fig_pred = mps.plot_predictions(
        str(key),
        dates_test, 
        y_test, 
        y_test_pred, 
        periode_limite=periode_limite,
    )
    plt.show()

    # projection des résidus et calcul du coefficient de dérive dans le temps
    fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
        str(key),
        dates_test, 
        y_test, 
        y_test_pred.values, 
        periode_limite=periode_limite
    )
    logging.info(f"Pente de la droite de régression des résidus dans le temps (dérive) : {model_res_coeff}")
    plt.show()